In [1]:
import spacy
# spacy.cli.download("nl_core_news_lg")
# nlp = spacy.load("nl_core_news_lg") 

#spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm") 

from pathlib import Path
from datasets import load_dataset

from src.lexicon_builder import *
from src.corpus_adaptors import *
from src.extractors import *

In [2]:
cut_off = 10
CCfile = Path(f'/Users/sabijn/Documents/PhD/code/storylm_p1_data/datasets/CommonCrawl/CC_unigram_lexicon_{cut_off}.csv')
CC_dataset = load_dataset('codymd/cc100_en_sample')
CC_dataset_text = CC_dataset['train']['text']

In [3]:
uni_gram_dep = load_or_build_lexicon(
    out_path=CCfile,
    builder_fn=lambda: build_lexicon(
        texts=iter_texts_from_dataset(CC_dataset_text, limit_docs=10),
        nlp=nlp,
        extractor=extract_unigrams_from_doc,
        batch_size=1,
        n_process=1,
        key_names=("lemma","pos"),
    ),
    key_names=("lemma","pos"),
)

In [5]:
cutoff = 10
out_paths = {
    1: Path(f"CommonCrawl/CC_pos_unigram_{cutoff}.csv"),
    2: Path(f"CommonCrawl/CC_pos_bigram_{cutoff}.csv"),
    3: Path(f"CommonCrawl/CC_pos_trigram_{cutoff}.csv"),
}

dfs = load_or_build_pos_ngram_lexicons(
    out_paths=out_paths,
    texts_factory=lambda: iter_texts_from_dataset(CC_dataset_text, limit_docs=cutoff),
    nlp=nlp,
    orders=(1,2,3),
    batch_size=100,
    n_process=1,
)

df_uni, df_bi, df_tri = dfs[1], dfs[2], dfs[3]